# Head camera · YOLO11n 학습

7개 클래스: apple, banana, fanta_can, green_apple, mug, peach, plate.

학습 1,087장 / 검증 175장. 두 테스트 영상은 포함하지 않습니다. 현재 승인한 라벨 초안으로 학습합니다.

설정: GPU, 100 epochs, 이미지 크기 960, batch 8. 런타임 → 런타임 유형 변경에서 T4 GPU를 선택한 뒤 모두 실행하세요.

학습 완료 시 best.pt, last.pt와 로그·그래프를 포함한 ZIP 다운로드가 시작됩니다. 중간 체크포인트는 10 epoch 간격으로 저장됩니다.


In [ ]:
%pip -q install ultralytics==8.4.174
import torch
assert torch.cuda.is_available(), 'GPU 런타임을 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


In [ ]:
from pathlib import Path
import zipfile, hashlib, yaml
from google.colab import files

archive = Path('/content/head_camera_yolo11_dataset.zip')
if not archive.exists():
    print('head_camera_yolo11_dataset.zip 파일을 업로드하세요.')
    uploaded = files.upload()
    del uploaded
assert archive.exists(), '데이터셋 ZIP 파일이 필요합니다.'
with archive.open('rb') as f:
    assert hashlib.file_digest(f, 'sha256').hexdigest() == '897e2d22b3fe08e968a01307a7eaacb319693b5c8e98c3c1ce3748930811d2d0', '업로드된 데이터셋 파일이 준비한 파일과 다릅니다.'
dataset = Path('/content/head_camera_dataset')
dataset.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    z.extractall(dataset)
config = yaml.safe_load((dataset/'data.yaml').read_text())
config['path'] = str(dataset)
(dataset/'data.yaml').write_text(yaml.safe_dump(config, sort_keys=False))
for split, expected in [('train', 1087), ('val', 175)]:
    images = sorted((dataset/'images'/split).glob('*.jpg'))
    assert len(images) == expected
    assert all((dataset/'labels'/split/(p.stem+'.txt')).exists() for p in images)
    print(split, len(images), 'images')
print('7 classes:', config['names'])


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil, json
from google.colab import files

EPOCHS = 100
IMAGE_SIZE = 960
BATCH_SIZE = 8
model = YOLO('yolo11n.pt')
results = model.train(
    data='/content/head_camera_dataset/data.yaml',
    epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
    device=0, workers=2, seed=42, cache=False,
    project='/content/head_camera_training', name='yolo11n_head_camera',
    save=True, save_period=10, exist_ok=True,
)
run_dir = Path(model.trainer.save_dir)
print('Best weights:', run_dir/'weights/best.pt')
print('Last weights:', run_dir/'weights/last.pt')
artifact = shutil.make_archive('/content/head_camera_yolo11n_results', 'zip', run_dir)
print('Training complete. Downloading results:', artifact)
files.download(artifact)
